# 11 . Tokenise the Glasser52 version

Same as notebook 01, with one extra step. After selecting the 210 cortical rows, the
signals are collapsed to 52 parcels using the weight matrix from notebook 10, and the
geometry comes from `parcel_L_glasser52.npy` rather than the Brainnetome one.

Everything else is identical, so the 210 and 52 results differ only by parcellation.

In [13]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, h5py, torch
import scipy.signal
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni

DATA = '../data'; OUT = '../results_parcel'
CKPT_ROOT = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection'

MODEL   = 'tiny'          # 'tiny' or 'base'
SUFFIX  = 'glasser52'     # what notebook 10 saved
FS_ORIG, FS_TARGET, SEGMENT_SECS = 600, 256, 30
SEG_LEN = SEGMENT_SECS * FS_TARGET

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('device:', device, '| model:', MODEL, '| parcellation:', SUFFIX)

device: mps | model: tiny | parcellation: glasser52


In [14]:
# Geometry for the 52 parcels, plus the two things needed to get from Pooja's 246 rows
# down to 52. keep_idx picks the 210 cortical rows, then W averages them into parcels.
L        = np.load(f'{OUT}/parcel_L_{SUFFIX}.npy')            # (52, 6)
S        = np.load(f'{OUT}/parcel_S_{SUFFIX}.npy')            # (52,)
keep_idx = np.load(f'{OUT}/parcel_keep_idx_cortical.npy')     # (210,) rows out of 246
W        = np.load(f'{OUT}/glasser_W_counts.npy').astype(np.float32)  # (210, 52) voxel counts

assert L.shape == (52, 6) and W.shape == (len(keep_idx), 52), 'shape mismatch, rerun notebook 10'
print(f'geometry L {L.shape} | S {S.shape}')
print(f'mapping  keep_idx {keep_idx.shape} -> W {W.shape} -> {W.shape[1]} parcels')

pos_t   = torch.tensor(L, dtype=torch.float32).unsqueeze(0).to(device)
stype_t = torch.tensor(S, dtype=torch.int32).unsqueeze(0).to(device)

geometry L (52, 6) | S (52,)
mapping  keep_idx (210,) -> W (210, 52) -> 52 parcels


In [15]:
cfg  = json.load(open(f'{CKPT_ROOT}/{MODEL}/model_cfg.json'))
ckpt = torch.load(f'{CKPT_ROOT}/{MODEL}/BrainOmni.pt', map_location='cpu', weights_only=True)
model = BrainOmni(**cfg)
model.load_state_dict(ckpt, strict=False)
for p in model.parameters():
    p.requires_grad = False
model = model.to(device).eval()

def project_only(x, pos, stype):
    with torch.no_grad():
        tok, _ = model.tokenizer.tokenize(x, pos, stype, model.overlap_ratio)
        B, Cc, Wd, _ = tok.shape
        neuro = model.tokenizer.encoder.neuros.type_as(tok).detach().view(1, Cc, 1, -1)
        return model.projection(tok + neuro)

print('model loaded, frozen.')

model loaded, frozen.


In [16]:
def pca_merge(sig, W):
    """(210, T) regions -> (52, T) parcels via each parcel's first principal component.

    Sign invariant, which matters because Pooja's regions have arbitrary signs. See the
    diagnosis and comparison in notebook 10.
    """
    out = np.zeros((W.shape[1], sig.shape[1]), np.float32)
    for p in range(W.shape[1]):
        m = np.where(W[:, p] > 0)[0]
        X = sig[m] - sig[m].mean(1, keepdims=True)
        X = X / (X.std(1, keepdims=True) + 1e-12)
        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        comp = Vt[0] * S[0]
        if np.corrcoef(comp, X.mean(0))[0, 1] < 0:
            comp = -comp
        out[p] = comp
    return out

print('pca_merge ready, replaces weighted averaging.')

pca_merge ready, replaces weighted averaging.


In [17]:
def preprocess(sig):
    """(C, 36000) at 600 Hz -> two (C, 7680) segments at 256 Hz, z normalised per channel."""
    sig = scipy.signal.resample_poly(sig, up=32, down=75, axis=1).astype(np.float32)
    mean = sig.mean(axis=1, keepdims=True)
    std  = sig.std(axis=1, keepdims=True) + 1e-13
    sig  = (sig - mean) / std
    return sig[:, :SEG_LEN], sig[:, SEG_LEN:SEG_LEN*2]

def tokenize_file(path, label):
    out1, out2, ids = [], [], []
    with h5py.File(path, 'r') as f:
        n_subj = f['matrix'].shape[2]
        refs = f['radid'][:].flatten()
        for i in range(n_subj):
            sig = np.array(f['matrix'][:, :, i]).T      # (246, 36000)
            sig = sig[keep_idx]                         # (210, 36000)
            # Collapse to 52 parcels with PCA, not averaging. Pooja's regions carry
            # arbitrary signs, so averaging cancels signal (see notebook 10). PC1 is
            # invariant to those signs, and matches what osl-ephys does for MEG-GPT.
            sig = pca_merge(sig, W)                      # (52, 36000)
            s1, s2 = preprocess(sig)
            for seg, store in ((s1, out1), (s2, out2)):
                x = torch.tensor(seg, dtype=torch.float32).unsqueeze(0).to(device)
                store.append(project_only(x, pos_t, stype_t).squeeze(0).cpu().numpy())
            ids.append(''.join(chr(c) for c in np.asarray(f[refs[i]]).flatten()))
            if (i + 1) % 10 == 0:
                print(f'  {os.path.basename(path)}: {i+1}/{n_subj}', flush=True)
    return np.stack(out1), np.stack(out2), ids, [label] * n_subj

print('helpers ready.')

helpers ready.


In [18]:
import time
t0 = time.time()
ad1, ad2, ad_ids, ad_lab = tokenize_file(f'{DATA}/ts48/tsAD.mat',   1)
print(f'AD done: {ad1.shape} ({(time.time()-t0)/60:.1f} min)', flush=True)

ct1, ct2, ct_ids, ct_lab = tokenize_file(f'{DATA}/ts48/tsCONT.mat', 0)
print(f'CONT done: {ct1.shape} ({(time.time()-t0)/60:.1f} min total)', flush=True)

  tsAD.mat: 10/77
  tsAD.mat: 20/77
  tsAD.mat: 30/77
  tsAD.mat: 40/77
  tsAD.mat: 50/77
  tsAD.mat: 60/77
  tsAD.mat: 70/77
AD done: (77, 16, 160, 256) (24.7 min)
  tsCONT.mat: 10/84
  tsCONT.mat: 20/84
  tsCONT.mat: 30/84
  tsCONT.mat: 40/84
  tsCONT.mat: 50/84
  tsCONT.mat: 60/84
  tsCONT.mat: 70/84
  tsCONT.mat: 80/84
CONT done: (84, 16, 160, 256) (54.8 min total)


In [19]:
proj1  = np.concatenate([ad1, ct1]).astype(np.float32)
proj2  = np.concatenate([ad2, ct2]).astype(np.float32)
labels = np.array(ad_lab + ct_lab, dtype=np.int64)
subj   = np.array(ad_ids + ct_ids)

print('proj_out1:', proj1.shape, '| proj_out2:', proj2.shape)
print('labels:', labels.shape, '| AD', int((labels==1).sum()), '| Control', int((labels==0).sum()))
print('finite values only?', bool(np.isfinite(proj1).all() and np.isfinite(proj2).all()))
print('value range:', float(proj1.min()), 'to', float(proj1.max()))

# Cross check against the 210 region version. Same subjects in the same order, and the
# tokeniser collapses any channel count to the same shape, so these must match.
ref = f'{OUT}/parcel_proj_out1_{MODEL}_cortical.npy'
if os.path.exists(ref):
    r = np.load(ref, mmap_mode='r')
    print(f'\n210 region version: {r.shape}')
    print(f'same per subject shape? {r.shape[1:] == proj1.shape[1:]}')
    print(f'same subject count?    {r.shape[0] == proj1.shape[0]}')

np.save(f'{OUT}/parcel_proj_out1_{MODEL}_{SUFFIX}.npy', proj1)
np.save(f'{OUT}/parcel_proj_out2_{MODEL}_{SUFFIX}.npy', proj2)
np.save(f'{OUT}/parcel_labels_{SUFFIX}.npy', labels)
np.save(f'{OUT}/parcel_subject_ids_{SUFFIX}.npy', subj)
print(f'\nsaved to {OUT}/')
print('next: notebook 02 with SUFFIX = glasser52')

proj_out1: (161, 16, 160, 256) | proj_out2: (161, 16, 160, 256)
labels: (161,) | AD 77 | Control 84
finite values only? True
value range: -3.3341064453125 to 3.82275390625

210 region version: (161, 16, 160, 256)
same per subject shape? True
same subject count?    True

saved to ../results_parcel/
next: notebook 02 with SUFFIX = glasser52
